In [ ]:
from IPython.utils import io #Proporciona herramientas para capturar y suprimir las salidas (outputs).
import tqdm.notebook #Proporciona una barra de progreso interactiva para el notebook Jupyter.
import os, os.path, sys, random, subprocess #Proporciona funciones para interactuar con el sistema operativo, como manipular archivos y directorios.
total = 100
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
        pbar.update(20)
        !pip install matplotlib, seaborn, pandas
        '''Librerías gráficas'''
        import matplotlib.pyplot as plt
        from IPython.display import display, SVG, HTML #Muestra gráficos SVG en el notebook
        import seaborn as sns #Librería para visualización de datos estadísticos basada en matplotlib.
        !pip install rdkit
        from rdkit import Chem
        from rdkit.Chem import Draw
        from rdkit.Chem import Descriptors
        from rdkit.Chem import rdMolDescriptors
        pbar.update(40)
        '''Librerias de herramientas primarias:'''
        import pandas as pd #Librería para manipulación y análisis de datos tabulares en Python.
        %config Completer.use_jedi = False
        import json #Proporciona herramientas para trabajar con datos en formato JSON.
        pbar.update(30)
        '''Busqueda en ChEMBL (API)'''
        !pip install chembl_webresource_client
        from chembl_webresource_client.new_client import new_client #Proporciona un nuevo cliente para acceder a la API de ChEMBL.
        from chembl_webresource_client.utils import utils #Proporciona funciones de utilidad para trabajar con la API de ChEMBL.
        from pathlib import Path #Proporciona manipulación de rutas de archivos como clases independientes.
        '''Montar unidad de Google Drive'''
        from google.colab import drive #Proporciona herramientas para montar y acceder a Google Drive desde Google Colab.
        drive.mount("/content/drive")
        !pip install requests
        pbar.update(10)

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
url_DATA_G = "/content/drive/MyDrive/Doctorado/Set de Datos 2/Objetivo 2/1.3_CTD_datos_Curados.csv"
CTD_compounds = pd.read_csv(url_DATA_G, sep=",", encoding='latin1')

In [ ]:
CTD_compounds

,GeneSymbol,ChemicalName,Organism,Interaction,InteractionActions,PubMedIDs,ChemicalID
0,TP53,10-carboxymethyl-9-acridanone,Homo sapiens,10-carboxymethyl-9-acridanone affects the acti...,affects^activity,35435491,C026430
1,TP53,10-carboxymethyl-9-acridanone,Homo sapiens,10-carboxymethyl-9-acridanone metabolite affec...,affects^activity,35435491,C026430
2,TP53,10-decarbamoylmitomycin C,Homo sapiens,10-decarbamoylmitomycin C results in increased...,increases^stability,20536192,C067795
3,TP53,10-decarbamoylmitomycin C,Homo sapiens,[10-decarbamoylmitomycin C results in increase...,increases^expression|increases^stability,20536192,C067795
4,TP53,"1,10-phenanthroline",Homo sapiens,"1,10-phenanthroline analog affects the activit...",affects^activity,35435491,C025205
...,...,...,...,...,...,...,...
65334,TGFB3,"Water Pollutants, Chemical",Homo sapiens,Acetylcysteine inhibits the reaction [Water Po...,decreases^reaction|increases^expression|increa...,29408318,D014874
65335,TGFB3,"Water Pollutants, Chemical",Homo sapiens,Acetylcysteine inhibits the reaction [Water Po...,decreases^reaction|increases^expression,29408318,D014874
65336,TGFB3,"Water Pollutants, Chemical",Homo sapiens,"Water Pollutants, Chemical results in increase...",increases^expression|increases^secretion,29408318,D014874
65337,TGFB3,"Water Pollutants, Chemical",Homo sapiens,"Water Pollutants, Chemical results in increase...",increases^expression,29408318,D014874


In [ ]:
import pandas as pd
import requests
import time


In [ ]:
def buscar_en_pubchem(nombre):
    base_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

    # Primero, buscamos el CID del compuesto
    search_url = f"{base_url}/compound/name/{nombre}/cids/JSON"
    response = requests.get(search_url)

    if response.status_code == 200:
        data = response.json()
        if 'IdentifierList' in data and 'CID' in data['IdentifierList']:
            cid = data['IdentifierList']['CID'][0]

            # Luego, obtenemos el SMILES canónico usando el CID
            property_url = f"{base_url}/compound/cid/{cid}/property/CanonicalSMILES/JSON"
            prop_response = requests.get(property_url)

            if prop_response.status_code == 200:
                prop_data = prop_response.json()
                if 'PropertyTable' in prop_data and 'Properties' in prop_data['PropertyTable']:
                    return prop_data['PropertyTable']['Properties'][0]['CanonicalSMILES']

    print(f"No se encontraron resultados para: {nombre}")
    return None

def procesar_compuestos(df, columna_nombre):
    smiles_dict = {}
    total = len(df[columna_nombre].unique())
    for i, nombre in enumerate(df[columna_nombre].unique(), 1):
        print(f"Buscando: {nombre} ({i}/{total})")
        smiles = buscar_en_pubchem(nombre)
        smiles_dict[nombre] = smiles
        time.sleep(0.2)  # Para respetar los límites de la API de PubChem
    return smiles_dict


In [ ]:
# Procesar los compuestos
smiles_resultados = procesar_compuestos(CTD_compounds, 'ChemicalName')

# Añadir la columna SMILES al DataFrame original
CTD_compounds['SMILES'] = CTD_compounds['ChemicalName'].map(smiles_resultados)

# Imprimir resumen de resultados
print("\nResumen de resultados:")
print(f"Total de compuestos: {len(CTD_compounds)}")
print(f"Compuestos con SMILES encontrados: {CTD_compounds['SMILES'].notna().sum()}")
print(f"Compuestos sin SMILES (NA): {CTD_compounds['SMILES'].isna().sum()}")

Buscando: 10-carboxymethyl-9-acridanone (1/4033)
Buscando: 10-decarbamoylmitomycin C (2/4033)
Buscando: 1,10-phenanthroline (3/4033)
Buscando: 11,11'-dideoxyverticilin (4/4033)
No se encontraron resultados para: 11,11'-dideoxyverticilin
Buscando: 1,2,5,6-dibenzanthracene (5/4033)
Buscando: 1,2-bis(isothiazol-5-yl)disulfane (6/4033)
Buscando: 1',2'-dihydrorotenone (7/4033)
Buscando: 1,2-dioleoyl-3-phosphoethanolamine-n-(poly(ethyleneglycol))-hydroxy succinamide (8/4033)
No se encontraron resultados para: 1,2-dioleoyl-3-phosphoethanolamine-n-(poly(ethyleneglycol))-hydroxy succinamide
Buscando: 1,2-distearoyllecithin (9/4033)
Buscando: 1,2-distearoyl-sn-glycero-3-phosphoethanolamine-N-methoxy-poly(ethylene glycol 2000) (10/4033)
No se encontraron resultados para: 1,2-distearoyl-sn-glycero-3-phosphoethanolamine-N-methoxy-poly(ethylene glycol 2000)
Buscando: 12-hydroxyellipticine (11/4033)
No se encontraron resultados para: 12-hydroxyellipticine
Buscando: 13-hydroxyellipticine (12/4033)
No 

In [ ]:
# Mostrar las primeras filas del DataFrame actualizado
print("\nPrimeras filas del DataFrame actualizado:")
print(CTD_compounds.head())


Primeras filas del DataFrame actualizado:
  GeneSymbol                   ChemicalName      Organism  \
0       TP53  10-carboxymethyl-9-acridanone  Homo sapiens   
1       TP53  10-carboxymethyl-9-acridanone  Homo sapiens   
2       TP53      10-decarbamoylmitomycin C  Homo sapiens   
3       TP53      10-decarbamoylmitomycin C  Homo sapiens   
4       TP53            1,10-phenanthroline  Homo sapiens   

                                         Interaction  \
0  10-carboxymethyl-9-acridanone affects the acti...   
1  10-carboxymethyl-9-acridanone metabolite affec...   
2  10-decarbamoylmitomycin C results in increased...   
3  [10-decarbamoylmitomycin C results in increase...   
4  1,10-phenanthroline analog affects the activit...   

                         InteractionActions PubMedIDs ChemicalID  \
0                          affects^activity  35435491    C026430   
1                          affects^activity  35435491    C026430   
2                       increases^stability  2053

In [ ]:
# Guardar el DataFrame actualizado en un nuevo archivo CSV
CTD_compounds.to_csv("CTD_compounds_smiles.csv", index=False)
print("\nResultados guardados en 'CTD_compounds_con_smiles.csv'")




Resultados guardados en 'CTD_compounds_con_smiles.csv'


In [ ]:
CTD_compounds

,GeneSymbol,ChemicalName,Organism,Interaction,InteractionActions,PubMedIDs,ChemicalID,SMILES
0,TP53,10-carboxymethyl-9-acridanone,Homo sapiens,10-carboxymethyl-9-acridanone affects the acti...,affects^activity,35435491,C026430,C1=CC=C2C(=C1)C(=O)C3=CC=CC=C3N2CC(=O)O
1,TP53,10-carboxymethyl-9-acridanone,Homo sapiens,10-carboxymethyl-9-acridanone metabolite affec...,affects^activity,35435491,C026430,C1=CC=C2C(=C1)C(=O)C3=CC=CC=C3N2CC(=O)O
2,TP53,10-decarbamoylmitomycin C,Homo sapiens,10-decarbamoylmitomycin C results in increased...,increases^stability,20536192,C067795,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2CO)OC)N4)N
3,TP53,10-decarbamoylmitomycin C,Homo sapiens,[10-decarbamoylmitomycin C results in increase...,increases^expression|increases^stability,20536192,C067795,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2CO)OC)N4)N
4,TP53,"1,10-phenanthroline",Homo sapiens,"1,10-phenanthroline analog affects the activit...",affects^activity,35435491,C025205,C1=CC2=C(C3=C(C=CC=N3)C=C2)N=C1
...,...,...,...,...,...,...,...,...
65334,TGFB3,"Water Pollutants, Chemical",Homo sapiens,Acetylcysteine inhibits the reaction [Water Po...,decreases^reaction|increases^expression|increa...,29408318,D014874,None
65335,TGFB3,"Water Pollutants, Chemical",Homo sapiens,Acetylcysteine inhibits the reaction [Water Po...,decreases^reaction|increases^expression,29408318,D014874,None
65336,TGFB3,"Water Pollutants, Chemical",Homo sapiens,"Water Pollutants, Chemical results in increase...",increases^expression|increases^secretion,29408318,D014874,None
65337,TGFB3,"Water Pollutants, Chemical",Homo sapiens,"Water Pollutants, Chemical results in increase...",increases^expression,29408318,D014874,None
